In [1]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
# Doğru import bu şekildedir:
from torch.utils.data import DataLoader, Dataset 
import albumentations as A
from albumentations.pytorch import ToTensorV2
from mimari import get_segmentation_model, get_loss_function

# Cihaz ve Dosya Yolları (Kendi klasörlerinle güncelle)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bebek_train_img = r"C:\Users\Ayberk\Desktop\BebekData\Train_Image"
bebek_train_mask = r"C:\Users\Ayberk\Desktop\BebekData\Train_Mask"
bebek_val_img = r"C:\Users\Ayberk\Desktop\BebekData\Val_Image"
bebek_val_mask = r"C:\Users\Ayberk\Desktop\BebekData\Val_Mask"

# Yetişkin verisinden elde ettiğimiz en iyi ağırlık
yetiskin_weights_path = "best_yetiskin_model.pth"

C:\Users\Ayberk\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================================
# 1. AYARLAR VE YOLLAR
# ==========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Klasör yollarını bebek datalarına göre güncelle
TRAIN_IMG_DIR = r"C:\Users\Ayberk\Desktop\BebekData\Train_Image"
TRAIN_MASK_DIR = r"C:\Users\Ayberk\Desktop\BebekData\Train_Mask"
VAL_IMG_DIR   = r"C:\Users\Ayberk\Desktop\BebekData\Val_Image"
VAL_MASK_DIR   = r"C:\Users\Ayberk\Desktop\BebekData\Val_Mask"

YETISKIN_WEIGHTS = "best_yetiskin_model.pth" # Yetişkin eğitiminden gelen dosya
SAVE_PATH = "final_bebek_model.pth"

BATCH_SIZE = 8
NUM_EPOCHS = 40
LEARNING_RATE = 1e-5 # Fine-tuning için düşük oran

In [3]:
# ==========================================================
# 2. DATASET SINIFI
# ==========================================================
class FundusDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        # Sadece resim dosyalarını listele
        self.images = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_name = self.images[index]
        img_path = os.path.join(self.img_dir, img_name)
        
        # --- Dinamik Maske Arama ---
        # Önce resimle aynı isimde ara
        mask_path = os.path.join(self.mask_dir, img_name)
        
        # Eğer yoksa, uzantıyı .png yaparak dene (Yaygın bir durumdur)
        if not os.path.exists(mask_path):
            mask_name_png = os.path.splitext(img_name)[0] + ".png"
            mask_path = os.path.join(self.mask_dir, mask_name_png)

        # Hala yoksa hata fırlatmadan önce kontrol et
        if not os.path.exists(mask_path):
             raise FileNotFoundError(f"Maske dosyası hiçbir uzantıyla bulunamadı: {img_name} "
                                     f"\nAranan konum: {self.mask_dir}")

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = np.float32(mask > 127)

        if self.transform:
            augmentations = self.transform(image=image, mask=mask)
            image = augmentations["image"]
            mask = augmentations["mask"]

        return image, mask.unsqueeze(0)

In [4]:
# ==========================================================
# 3. TRANSFORM VE DATALOADER
# ==========================================================
train_transform = A.Compose([
    A.Resize(384, 384),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(384, 384),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

train_ds = FundusDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform)
val_ds = FundusDataset(VAL_IMG_DIR, VAL_MASK_DIR, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

In [5]:
# ==========================================================
# 4. MODEL, LOSS VE OPTIMIZER
# ==========================================================
model = get_segmentation_model().to(device)

# Yetişkin ağırlıklarını yükle
if os.path.exists(YETISKIN_WEIGHTS):
    model.load_state_dict(torch.load(YETISKIN_WEIGHTS))
    print(f"BAŞARILI: {YETISKIN_WEIGHTS} ağırlıkları yüklendi.")
else:
    print("UYARI: Yetişkin ağırlıkları bulunamadı, eğitime sıfırdan başlanıyor.")

criterion = get_loss_function()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


BAŞARILI: best_yetiskin_model.pth ağırlıkları yüklendi.


In [6]:
# ==========================================================
# 5. FINE-TUNING DÖNGÜSÜ
# ==========================================================
# ==========================================================
# 5. EĞİTİM DÖNGÜSÜ
# ==========================================================
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False)
    
    for images, masks in train_loop:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        
        # Standart float32 eğitim (AMP yok)
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())
    
    # Validation
    model.eval()
    val_loss = 0
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]", leave=False)
    
    with torch.no_grad():
        for images, masks in val_loop:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            val_loop.set_postfix(loss=loss.item())
            
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"--> Başarı Artışı! Model kaydedildi: {SAVE_PATH}")
    print("-" * 40)

print("Eğitim tamamlandı!")

Epoch 1 | Train Loss: 0.4969 | Val Loss: 0.4954
--> Başarı Artışı! Model kaydedildi: final_bebek_model.pth
----------------------------------------


Epoch 2 | Train Loss: 0.4945 | Val Loss: 0.4953
--> Başarı Artışı! Model kaydedildi: final_bebek_model.pth
----------------------------------------


Epoch 3 | Train Loss: 0.4944 | Val Loss: 0.4953
--> Başarı Artışı! Model kaydedildi: final_bebek_model.pth
----------------------------------------


Epoch 4 | Train Loss: 0.4943 | Val Loss: 0.4947
--> Başarı Artışı! Model kaydedildi: final_bebek_model.pth
----------------------------------------


Epoch 5 | Train Loss: 0.4933 | Val Loss: 0.4941
--> Başarı Artışı! Model kaydedildi: final_bebek_model.pth
----------------------------------------


...

Epoch 36 | Train Loss: 0.4906 | Val Loss: 0.4899
----------------------------------------


Epoch 37 | Train Loss: 0.4908 | Val Loss: 0.4899
--> Başarı Artışı! Model kaydedildi: final_bebek_model.pth
----------------------------------------


Epoch 38 | Train Loss: 0.4902 | Val Loss: 0.4899
----------------------------------------


Epoch 39 | Train Loss: 0.4911 | Val Loss: 0.4898
--> Başarı Artışı! Model kaydedildi: final_bebek_model.pth
----------------------------------------


Epoch 40 | Train Loss: 0.4910 | Val Loss: 0.4898
----------------------------------------
Eğitim tamamlandı!
